In [4]:
%matplotlib widget
import csv
import os
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from dataclasses import dataclass

# ============================================================
# 1. STYLE & OBJECT CONFIGURATION
# ============================================================
@dataclass
class Spectrum:
    N: str                  # Nombre o Label original
    X: np.ndarray           # Wavelength (nm)
    Y: np.ndarray           # Absorbance / Intensity
    color: str = None       
    linestyle: str = None   

    def copy(self):
        return Spectrum(
            N=self.N, 
            X=self.X.copy(), 
            Y=self.Y.copy(), 
            color=self.color,         
            linestyle=self.linestyle  
        )

poster_style = {
    "figsize": (8, 5),          
    "dpi": 150,                 
    "font_size": 11,
    "title_size": 14,
    "label_size": 12,
    "legend_size": 10,
    "line_width": 2,
    "font_family": "Arial", 
    "grid": True,               
    "colors": ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442"], 
    "linestyles": ["-", "--", "-.", ":"]
}

# ============================================================
# 2. UV-VIS PLOTTING ENGINE (With Energy Axis Support)
# ============================================================
def PlotSpectra(spectra_list, mode="WE", style=poster_style, 
                xlabel_W="Wavelength (nm)", xlabel_E="Energy (eV)", ylabel="Absorbance", title="Absorption Spectra", 
                xmin=None, xmax=None, ymin=None, ymax=None, 
                offset_step=0.0, save_path=None):
    
    fig, ax1 = plt.subplots(figsize=style["figsize"], dpi=style["dpi"])
    plt.rcParams.update({"font.size": style["font_size"], "font.family": style["font_family"]})
    
    default_colors = style.get("colors") or ["#000000"]
    default_linestyles = style.get("linestyles") or ["-"]
    
    for i, s in enumerate(spectra_list):
        c = getattr(s, 'color', None) or default_colors[i % len(default_colors)]
        ls = getattr(s, 'linestyle', None) or default_linestyles[i % len(default_linestyles)]
        y_stacked = s.Y + (i * offset_step)
        
        if mode in ["W", "WE"]:
            ax1.plot(s.X, y_stacked, label=s.N, linewidth=style["line_width"], color=c, linestyle=ls)
            ax1.set_xlabel(xlabel_W, fontsize=style["label_size"])
        elif mode in ["E", "EW"]:
            E_array = np.divide(1240.0, s.X, out=np.zeros_like(s.X), where=s.X != 0)
            ax1.plot(E_array, y_stacked, label=s.N, linewidth=style["line_width"], color=c, linestyle=ls)
            ax1.set_xlabel(xlabel_E, fontsize=style["label_size"])

    ax1.set_ylabel(ylabel, fontsize=style["label_size"])
    ax1.set_title(title, fontsize=style["title_size"])
    
    if xmin is not None or xmax is not None: ax1.set_xlim(xmin, xmax)
    if ymin is not None or ymax is not None: ax1.set_ylim(ymin, ymax)
    if style.get("grid", False): ax1.grid(True, alpha=0.3)

    # --- Ejes Secundarios (Energía <-> Wavelength) ---
    if mode in ["WE", "EW"]:
        def forward(x):
            with np.errstate(divide='ignore', invalid='ignore'):
                return np.where(x == 0, np.inf, 1240.0 / x)
        def inverse(x):
            with np.errstate(divide='ignore', invalid='ignore'):
                return np.where(x == 0, np.inf, 1240.0 / x)

        if mode == "WE":
            ax2 = ax1.secondary_xaxis('top', functions=(forward, inverse))
            ax2.set_xlabel(xlabel_E, fontsize=style["label_size"])
        elif mode == "EW":
            ax2 = ax1.secondary_xaxis('top', functions=(inverse, forward))
            ax2.set_xlabel(xlabel_W, fontsize=style["label_size"])

    ax1.legend(fontsize=style["legend_size"], ncol=1)
    fig.tight_layout()
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=style["dpi"], bbox_inches="tight")
        print(f"Saved: {save_path}")
        
    plt.show()

# ============================================================
# 3. ABSORPTION CSV LOADER (ROBUST VERSION)
# ============================================================
def read_absorption_csv(filename):
    samples = {}
    with open(filename, "r", newline="", encoding="utf-8-sig") as f:
        # Algunos exportadores usan punto y coma (;), si es tu caso, cámbialo a delimiter=";"
        reader = csv.reader(f, delimiter=",")
        
        # 1. Buscar la primera fila que no esté vacía (por si hay espacios en blanco al inicio)
        row1_names = None
        for row in reader:
            if any(cell.strip() for cell in row):
                row1_names = row
                break
                
        if not row1_names:
            raise ValueError(f"El archivo parece estar vacío.")
            
        # Saltamos la segunda fila (Wavelength (nm), Abs...)
        row2_headers = next(reader, [])

        # 2. Identificar dinámicamente en qué columna está cada muestra
        valid_samples = []
        for col_idx, cell in enumerate(row1_names):
            name = cell.strip()
            if name:  # Si la celda tiene texto, ahí empieza una muestra
                valid_samples.append((col_idx, name))
                samples[name] = ([], [])

        # 3. Leer los datos numéricos
        for row in reader:
            if not row or all(not cell.strip() for cell in row): continue

            for col_x, name in valid_samples:
                col_y = col_x + 1  # Asumimos que la absorbancia siempre está a la derecha del Wavelength
                
                if col_y < len(row):
                    x_str = row[col_x].strip()
                    y_str = row[col_y].strip()
                    if x_str and y_str:
                        try:
                            samples[name][0].append(float(x_str))
                            samples[name][1].append(float(y_str))
                        except ValueError:
                            continue

    # 4. Convertir a objetos Spectrum
    spectra_list = []
    for name, (x_list, y_list) in samples.items():
        if len(x_list) > 0:
            x_arr = np.array(x_list)
            y_arr = np.array(y_list)
            # Ordenamos por Wavelength para evitar que el gráfico dibuje líneas hacia atrás
            sort_idx = np.argsort(x_arr)
            spectra_list.append(Spectrum(N=name, X=x_arr[sort_idx], Y=y_arr[sort_idx]))
            
    return spectra_list

def load_all_absorption_spectra(folder_path, pattern="*.csv"):
    folder = Path(folder_path)
    
    # Usamos .rglob() para buscar recursivamente en todas las subcarpetas
    files = list(folder.rglob("*.csv")) + list(folder.rglob("*.CSV"))
    
    if not files:
        print(f"❌ ATENCIÓN: No se encontró ningún archivo CSV en la ruta ni en sus subcarpetas:\n{folder_path}")
        return {}
        
    global_spectra_dict = {}
    for file in sorted(files):
        try:
            file_spectra = read_absorption_csv(file)
            print(f"✅ Leído [{file.parent.name}]: {file.name} -> {len(file_spectra)} espectros.")
            
            for s in file_spectra:
                # Llave única que incluye la subcarpeta, el archivo y la muestra
                unique_key = f"{file.parent.name}_{file.stem}_{s.N}"
                global_spectra_dict[unique_key] = s 
                
        except Exception as e:
            print(f"⚠️ Error saltando {file.name}: {e}")
            
    return global_spectra_dict
# ============================================================
# 4. EXECUTION & SELECTION
# ============================================================
FOLDER_PATH = r"H:\UAntwerp\2. Paper on DWCNTs - Shivani KIT\Data\Absorption" # Cambia esto a tu carpeta real

# 1. Cargamos TODOS los espectros
spectra_dict = load_all_absorption_spectra(FOLDER_PATH, "*.csv")

print(f"Archivos procesados. Se encontraron {len(spectra_dict)} espectros únicos:")
for key in spectra_dict.keys():
    print(f"'{key}',")
print("\n")


✅ Leído [20240531]: KIT_Samples.csv -> 14 espectros.
✅ Leído [20240531]: KIT_Samples.csv -> 14 espectros.
✅ Leído [20240613]: KIT_ConvertedSamples.csv -> 4 espectros.
✅ Leído [20240613]: KIT_ConvertedSamples.csv -> 4 espectros.
✅ Leído [20240614]: KIT_FilmConvertedSamples.csv -> 7 espectros.
✅ Leído [20240614]: KIT_FilmConvertedSamples.csv -> 7 espectros.
✅ Leído [20250305]: DWCNT_Megasonicated_KIT.csv -> 2 espectros.
✅ Leído [20250305]: DWCNT_Megasonicated_KIT.csv -> 2 espectros.
✅ Leído [20250305]: TDAE_Cryostat.csv -> 2 espectros.
✅ Leído [20250305]: TDAE_Cryostat.csv -> 2 espectros.
✅ Leído [20250305]: TDAE_Cryostat_NoBaseline.csv -> 2 espectros.
✅ Leído [20250305]: TDAE_Cryostat_NoBaseline.csv -> 2 espectros.
✅ Leído [20250321]: LastDialisysAndSamplePostDial.csv -> 3 espectros.
✅ Leído [20250321]: LastDialisysAndSamplePostDial.csv -> 3 espectros.
Archivos procesados. Se encontraron 34 espectros únicos:
'20240531_KIT_Samples_Baseline 100%T',
'20240531_KIT_Samples_T4ConvertedPellet'

In [14]:
spectra_config = [
    # =========================================================
    # 1. KIT SAMPLES (2024-05-31)
    # =========================================================
     #('20240531_KIT_Samples_T1Converted',     'T1 Converted',      '#0072B2', '-'),
     #('20240531_KIT_Samples_T4Converted',     'T4 Converted',      '#D55E00', '-'),
     #('20240531_KIT_Samples_T4ConvertedPellet','T4 Pellet',        '#CC79A7', '-'),
     #('20240531_KIT_Samples_T4ConvertedPelletB','T4 Pellet B',     '#E69F00', '-'),

    # =========================================================
    # 2. CONVERTED SAMPLES (2024-06-13)
    # =========================================================
     #('20240613_KIT_ConvertedSamples_T1S',            'T1S',               '#0072B2', '-'),
     #('20240613_KIT_ConvertedSamples_T4S',            'T4S',               '#D55E00', '-'),
     #('20240613_KIT_ConvertedSamples_T4P',            'T4P',               '#009E73', '-'),

    # =========================================================
    # 3. FILM CONVERTED SAMPLES (2024-06-14)
    # =========================================================
     #('20240614_KIT_FilmConvertedSamples_FilmT1S',             'Film T1S (SWCNT)', '#0072B2', '-'),
     #('20240614_KIT_FilmConvertedSamples_FilmT1SConverted',     'Film T1S Conv',    '#0072B2', '--'),
     #('20240614_KIT_FilmConvertedSamples_FilmT2S',             'Film T2S (SWCNT)', '#D55E00', '-'),
     #('20240614_KIT_FilmConvertedSamples_FilmT2SConverted',     'Film T2S Conv',    '#D55E00', '--'),
     #('20240614_KIT_FilmConvertedSamples_FilmT9M',             'Film T9M (SWCNT)', '#009E73', '-'),
     #('20240614_KIT_FilmConvertedSamples_FilmT9MConverted',     'Film T9M Conv',    '#009E73', '--'),
      
    # =========================================================
    # 5. DIALYSIS & POST-DIAL SAMPLES (2025-03-21)
    # =========================================================
    #('20250321_LastDialisysAndSamplePostDial_DWCNT_Megasonicated_KIT_Dial', 'Megasonicated Dial', '#CC79A7', '-'),
]

Selection = []
for key, name, col, ls in spectra_config:
    if key in spectra_dict:
        s = spectra_dict[key].copy()
        s.N = name
        s.color = col
        s.linestyle = ls
        Selection.append(s)
    else:
        print(f"Advertencia: El espectro '{key}' no existe.")


# --- PIPELINE DE PROCESAMIENTO (Opcional) ---
# Selection = TrimSpectra(Selection, xmin=400, xmax=1600)
# Selection = SubtractMinimum(Selection, xmin=1400, xmax=1550)
# Selection = NormalizeSpectra(Selection, xmin=900, xmax=1100, mode="M")

# 3. GRAFICAR
if len(Selection) > 0:
    PlotSpectra(
        spectra_list=Selection,
        mode="WE",          # Muestra nm abajo y eV arriba
        style=poster_style,
        title="Absorption Spectra",
        offset_step=0.0     # Cambia esto a 0.5 o 1.0 si quieres separar los espectros
    )